<a href="https://colab.research.google.com/github/gburtch/BA865-2026/blob/main/8_Apr%2014/8.7%20-%20Pre-trained%20LLM%20with%20RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Retrieval Augmented Generation with LLMs**

Before starting this tutorial, make sure you have completed the following steps:

Get access to Gemma on HuggingFace (login when prompted using teh cell below, and go created an API token that has Read access).

Select a Colab runtime with sufficient resources to run the Gemma model size you want to run.

Generate and configure a Kaggle username and API key.

In [1]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


#Import Libraries and Load an LLM

We will install updated versions of Keras hub, the latest pre-release updates for Keras (keras-nightly), and pypdf2, which we can use to extract text from PDF files (our reference database in our RAG).

In [ ]:
!pip install --upgrade --quiet keras-hub-nightly keras-nightly
!pip install --quiet pypdf2

import tensorflow as tf
import keras_hub
import keras_nlp
import numpy as np
import pandas as pd
import keras
import keras_hub
import os
from PyPDF2 import PdfReader
from sklearn.metrics.pairwise import cosine_similarity # We will use cosine similarity to implement our document lookup.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.3 MB/s eta 0:00:00


Let's load a pre-trained LLM from Keras Hub. We will use Gemma 4 2B Instruct.

NOTE: You may need to accept a Gemma license on Hugging Face before downloading the model.

In [ ]:
# Load a Model with float16 precision (quantization is happening!). This step may take some time.
gemma_chat = keras_hub.models.Gemma4CausalLM.from_preset(
    "hf://google/gemma-4-E2B-it",
    dtype="float16"
)

# Extract the tokenizer from the model for later use (embeddings).
tokenizer = gemma_chat.preprocessor.tokenizer

We will ask Gemma 4 to answer a few questions for us about Moby Dick.

NOTE: it will take potentially 4-5 minutes to get output from Gemma 4 when you first prompt it, while it's getting setup / compiled in memory.

In [ ]:
# Helper to wrap a prompt in Gemma 4's chat template.
def format_prompt(text):
    return f"<start_of_turn>user\n{text}<end_of_turn>\n<start_of_turn>model\n"

# Here are our prompts. Note that you cannot trust confidence scores to be meaningful!
prompt_1 = "What was the name of the first mate on Ahab's ship in Moby Dick? On a scale from 1-100, how confident are you?"
prompt_2 = "Who was the third mate in Moby Dick? On a scale from 1-100, how confident are you?"
prompt_3 = "What was the name of the ship's carpenter in Moby Dick? On a scale from 1-100, how confident are you?"

# This one it gets right out of its pre-training!
print(gemma_chat.generate(format_prompt(prompt_1), max_length=256),"\n")

# Gemma won't answer, or gets it wrong. Note that the correct answer is 'Flask', but its mentioned rarely in the book.
print(gemma_chat.generate(format_prompt(prompt_2), max_length=256),"\n")

# This one it gets confidently wrong!!! The carpenter has no name in Moby Dick. He is only referred to as 'the Carpenter'.
# Gemma hallucinates the answer.
print(gemma_chat.generate(format_prompt(prompt_3), max_length=256),"\n")

<start_of_turn>user
What was the name of the first mate on Ahab's ship in Moby Dick? On a scale from 1-100, how confident are you?<end_of_turn>
<start_of_turn>model
The first mate on Ahab's ship in Moby Dick was Starbuck.
On a scale from 1-100, I am 100% confident in this answer.
<end_of_turn><turn|> 

<start_of_turn>user
Who was the third mate in Moby Dick? On a scale from 1-100, how confident are you?<end_of_turn>
<start_of_turn>model
The third mate in Moby Dick was **Stubb**.
On a scale from 1-100, I am **98** confident in this answer.
<end_of_turn><turn|> 

<start_of_turn>user
What was the name of the ship's carpenter in Moby Dick? On a scale from 1-100, how confident are you?<end_of_turn>
<start_of_turn>model
The ship's carpenter in Moby Dick was Tomahawk. On a scale from 1-100, I am 100% confident in this answer.
<end_of_turn><turn|> 



#Implement RAG with this Model

Write a function to extract the text from the PDF file.

In [ ]:
import re

# 350-token chunks of mutually exclusive text
def process_pdf(pdf_path, chunk_size=350, overlap=100):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + " "

    # Preprocessing
    text = re.sub(r'(?:^|\s)[^a-zA-Z\s]{1,3}(?:\s|$)', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text) and text[end] != ' ':
            end = text.rfind(' ', start, end) + 1
        chunk = text[start:end]

        # Skip chunks where the average word length is too short (likely LaTeX junk)
        words = chunk.split()
        if len(words) > 0 and np.mean([len(w) for w in words]) > 2.5:
            chunks.append(chunk)

        if end == len(text):
            break
        start = end - overlap

    return chunks

### A Note on Text Preprocessing

The `process_pdf()` function removes punctuation to clean noisy PDF text. We preserve apostrophes and hyphens to avoid corrupting names and possessives (e.g., "Ahab's" staying intact rather than becoming "Ahab s"). In production RAG systems, text preprocessing is critical — too aggressive cleaning destroys meaning, while too little leaves noise that hurts retrieval quality.

Let's find a copy of Moby Dick that we can chunk and use for RAG.

In [ ]:
# Download PDF of Moby Dick and save it locally for processing.
!wget -O moby_dick.pdf https://uberty.org/wp-content/uploads/2015/12/herman-melville-moby-dick.pdf

pdf_path = "moby_dick.pdf"

# Chunk size 100, 50 token overlap across chunks.
chunks = process_pdf(pdf_path, chunk_size=350, overlap=100)

print(chunks[5])

--2026-04-13 02:15:23--  https://uberty.org/wp-content/uploads/2015/12/herman-melville-moby-dick.pdf
Resolving uberty.org (uberty.org)... 68.66.200.199
Connecting to uberty.org (uberty.org)|68.66.200.199|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1544566 (1.5M) [application/pdf]
Saving to: ‘moby_dick.pdf’

moby_dick.pdf       100%[===================>]   1.47M  2.28MB/s    in 0.6s    

2026-04-13 02:15:24 (2.28 MB/s) - ‘moby_dick.pdf’ saved [1544566/1544566]

bb kills a whale LIX. e dart LX. e crotch LXI. Stubb’s supper LXII. e whale as a dish iii LXIII. Cutting in LXIV. e blanket LXV. e funeral LXVI. e sphynx LXVII. e monkey rope LXVIII. Stubb and Flask kill a right whale; and then have over him LXIX. e sperm whale’s head—contrasted view LXX. e battering-ram LXXI. e great Heidelburgh tun 


Let's look for some chunks that mention the third mate and the carpenter.

In [ ]:
# Find the first text chunk that mentions the 'third mate'...

third_mate_indices = []
for i,chunk in enumerate(chunks):
    if 'third mate' in chunk:
        third_mate_indices.append(i)

# Flask is referenced explicitly as third mate exactly one place in the book!
print(third_mate_indices)
print(chunks[third_mate_indices[0]])

[906, 907]
rless mortals who have died exhaling it; and as in time of the cholera, some people go about with a camphorated handkerchief to their mouths; so, likewise, against all mortal tribulations, Stubb’s tobacco smoke might have operated as a sort of disinfeing agent. e third mate was Flask, a native of Tisbury, in Martha’s Vineyard. A short, stout, 


We need to construct embeddings from our prompt and from our text chunks to be able to perform the lookup between our prompt and the recipe text chunks. We will use Sentence-BERT embeddings for this purpose. They are fine-tuned / well-suited for textual lookup via embedding cosine comparisons.

In [ ]:
!pip install --quiet sentence-transformers
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed all chunks
chunk_embeddings = embed_model.encode(chunks, show_progress_bar=True, batch_size=32)
rag_df = pd.DataFrame({'chunk': chunks, 'embedding': chunk_embeddings.tolist()})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/136 [00:00<?, ?it/s]

Here are our document chunk embeddings...

In [ ]:
chunk_index = 100

# Directly access and print the first chunk
print(f'CHUNK TEXT: \n\n{chunks[chunk_index]} \n')

# Count words directly
print(f'CHUNK DETAILS: \n\nThe chunk contains roughly {len(chunks[chunk_index].split(" "))} words (based on white spaces).')

# Look at embedding shape directly
print(f'Its embedding has {len(rag_df.loc[chunk_index,'embedding'])} dimensions.\n')

# Print vector representation directly
print(f'VECTOR REPRESENTATION:\n')
print(rag_df.loc[chunk_index,'embedding'])

CHUNK TEXT: 

e rest were plain. But stop; does it not bear a faint resemblance to a gigantic ﬁsh? even the great Leviathan himself? In fa, the artist’s design seemed this: a ﬁnal theory of my own, partly based upon the aggregated opinions of many aged persons with whom I conversed upon the subje. e piure represents a Cape-Horner in a great hurricane; the  

CHUNK DETAILS: 

The chunk contains roughly 63 words (based on white spaces).
Its embedding has 384 dimensions.

VECTOR REPRESENTATION:

[-0.0003169880947098136, 0.12437215447425842, 0.055712152272462845, 0.013337509706616402, 0.006499309092760086, -0.03597152978181839, 0.026334835216403008, 0.04491468891501427, -0.0010718252742663026, 0.04150401055812836, -0.05780238285660744, -0.026670269668102264, 0.03207114338874817, -0.030926937237381935, -0.06737687438726425, -0.03072279505431652, 0.004001697991043329, -0.08300398290157318, -9.167050302494317e-05, 0.08561623841524124, 0.047861356288194656, 0.03038419410586357, -0.02427828

We can save the embeddings to a pickle file for later use (or load a previously stored file)...

In [ ]:
import pandas as pd

# Save the embeddings to a pickle file for later use (or load a previously stored file)
rag_df.to_pickle('rag_df.pkl')
#rag_df = pd.read_pickle('rag_df.pkl')

Now we need a method to perform a top-k similar chunk lookup (relative to the prompt).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def rag_lookup(query, rag_data, top_k=2):
    query_embedding = embed_model.encode([query])
    chunk_embs = np.vstack(rag_data['embedding'].values)
    similarities = cosine_similarity(query_embedding, chunk_embs)[0]

    temp_df = rag_data.copy()
    temp_df['similarity'] = similarities
    results = temp_df.sort_values('similarity', ascending=False).head(top_k)
    return results

# RAG prompt creation
def create_rag_prompt(query, rag_data, top_k=2):
    results = rag_lookup(query, rag_data, top_k)
    prompt = f"Answer the following question: {query}\n\n"
    prompt += "Please refer to the following segments of text, drawn from relevant information sources, as you answer:\n"
    for i, (_, row) in enumerate(results.iterrows()):
        prompt += f"\n--- Source {i+1} ---\n{row['chunk']}\n"
    return prompt

rag_prompt = create_rag_prompt("Who was the third mate in Moby Dick?", rag_df, top_k=3)
print(rag_prompt)

Answer the following question: Who was the third mate in Moby Dick?

Please refer to the following segments of text, drawn from relevant information sources, as you answer:

--- Source 1 ---
infeing agent. e third mate was Flask, a native of Tisbury, in Martha’s Vineyard. A short, stout, ruddy young fellow, very pugnacious concerning whales, who somehow seemed to think that the great Leviathans had personally and hereditarily aﬀronted him; and therefore it was a sort of point of honor with him, to destroy them whenever encountered. 

--- Source 2 ---
b,” said Starbuck, who, with Stubb and Flask, had thus far been eyeing his superior with increasing surprise, but at last seemed struck with a thought which somewhat explained all the wonder. “Captain Ahab, I have heard of Moby Dick—but it was not Moby Dick that took oﬀ thy leg?” “Who told thee that?” cried Ahab; then pausing, “Aye, Starbuck; aye, 

--- Source 3 ---
against his captain’s leadership, unless some ordinary, pruden- tial, ci

And now we can see how the process would work, to provide additional context to the LLM in the prompt when obtaining its answer. Gemma 4 is an instruction-tuned model with a larger context window, so it should handle RAG context much better than Falcon did.

In [ ]:
# Without RAG
prompt = "Who was the third mate in Moby Dick?"
response_without_rag = None
response_without_rag = gemma_chat.generate(format_prompt(prompt), max_length=500)

# With RAG
rag_prompt = create_rag_prompt(prompt, rag_df, top_k=2)
response_with_rag = None
response_with_rag = gemma_chat.generate(format_prompt(rag_prompt), max_length=500)

# Print responses
print("WITHOUT RAG:")
print(response_without_rag)

print("\n\nWITH RAG:")
print(response_with_rag)

WITHOUT RAG:
<start_of_turn>user
Who was the third mate in Moby Dick?<end_of_turn>
<start_of_turn>model
The third mate in Moby Dick was **Stubb**.<turn|>


WITH RAG:
<start_of_turn>user
Answer the following question: Who was the third mate in Moby Dick?

Please refer to the following segments of text, drawn from relevant information sources, as you answer:

--- Source 1 ---
infeing agent. e third mate was Flask, a native of Tisbury, in Martha’s Vineyard. A short, stout, ruddy young fellow, very pugnacious concerning whales, who somehow seemed to think that the great Leviathans had personally and hereditarily aﬀronted him; and therefore it was a sort of point of honor with him, to destroy them whenever encountered. 

--- Source 2 ---
b,” said Starbuck, who, with Stubb and Flask, had thus far been eyeing his superior with increasing surprise, but at last seemed struck with a thought which somewhat explained all the wonder. “Captain Ahab, I have heard of Moby Dick—but it was not Moby Di

Now let's supplement RAG with scaled dotproduct cross attention. We are going to employ a 'weightless' (i.e., no Q, K, V weight matrices) cross-attention mechanism, to obviate the need for additional fine-tuning.    